In [1]:
import numpy as np
import pandas as pd
from scipy.special import kolmogorov
from scipy import stats

# Исходные данные
N = 100
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
alpha_level = 0.05

# Эмпирическая функция распределения
F_emp = np.array([sum(m[:i]) for i in range(len(m) + 1)]) / N

### a) Проверка гипотезы о согласии с равномерным распределением

In [2]:
# 1. Критерий Пирсона (хи-квадрат)
expected_freq = np.full(10, N / 10) 
chi2_stat_a, p_value_chi2_a = stats.chisquare(f_obs=m, f_exp=expected_freq)  # В тетради посчитали вручную, результаты совпадают

# 2. Критерий Колмогорова
F_th_uniform = np.arange(10) / 10

delta_a = np.sqrt(N) * np.max([
    max(np.abs(F_th_uniform[i] - F_emp[i]), np.abs(F_th_uniform[i] - F_emp[i+1])) 
    for i in range(10)
])

p_value_ks_a = kolmogorov(delta_a)

print("--- Равномерное распределение ---")
print(f"Пирсон (χ²): Статистика Δ = {chi2_stat_a:.4f}, p-value = {p_value_chi2_a:.4f}")
print(f"Колмогоров:  Статистика Δ = {delta_a:.4f}, p-value = {p_value_ks_a:.5f}\n")

if p_value_ks_a < alpha_level:
    print(f"p-value ({p_value_ks_a:.5f}) < α ({alpha_level})  ==>  H0 ОТВЕРГАЕТСЯ")
else:
    print(f"p-value ({p_value_ks_a:.5f}) >= α ({alpha_level}) ==>  Нет оснований отвергать H0")

--- Равномерное распределение ---
Пирсон (χ²): Статистика Δ = 16.4000, p-value = 0.0590
Колмогоров:  Статистика Δ = 1.4000, p-value = 0.03968

p-value (0.03968) < α (0.05)  ==>  H0 ОТВЕРГАЕТСЯ


### b) Проверка гипотезы о согласии с нормальным распределением

In [6]:
from scipy.optimize import minimize

expected_freq = np.full(10, N / 10) 
raw_sample = np.repeat(np.arange(10), m)
# Границы интервалов: (-inf, 1), [1, 2), ..., [9, +inf)
bounds = np.array([-np.inf] + list(range(1, 10)) + [np.inf])

# Функция отрицательного лог-правдоподобия (ее мы будем минимизировать, что эквивалентно максимизации правдоподобия)
def neg_log_likelihood(params):
    mu, sigma = params
    if sigma <= 0: 
        return np.inf
    
    # Считаем вероятности попадания в интервалы
    cdf_vals = stats.norm.cdf(bounds, mu, sigma)
    p = cdf_vals[1:] - cdf_vals[:-1]
    p = np.clip(p, 1e-12, 1.0) # Защита от логарифма нуля
    
    return -np.sum(m * np.log(p))

# В качестве начального приближения берем обычные выборочные оценки
initial_guess = [np.mean(raw_sample), np.std(raw_sample, ddof=1)]

# Численная оптимизация
mle_result = minimize(neg_log_likelihood, initial_guess, method='Nelder-Mead')
alpha_param, sigma_param = mle_result.x

segments = np.array([[-np.inf, 1]] + [[i, i + 1] for i in range(1, 9)] + [[9, np.inf]])

P = np.array([stats.norm.cdf(seg[1], alpha_param, sigma_param) - stats.norm.cdf(seg[0], alpha_param, sigma_param) for seg in segments])

delta_pearson_b = np.sum((N * P - m)**2 / (N * P))
p_value_chi2_b = stats.chi2.sf(delta_pearson_b, df=7)

print("--- Нормальное распределение (Пирсон) ---")
print(f"Оценки ОМПГ: μ ≈ {alpha_param:.4f}, σ ≈ {sigma_param:.4f}")
print(f"Статистика Δ ≈ {delta_pearson_b:.4f}, p-value ≈ {p_value_chi2_b:.5f}\n")

segments_str = ["(-∞; 1)"] + [f"[{i}; {i+1})" for i in range(1, 9)] + ["[9; +∞)"]
df = pd.DataFrame({
    "Интервал п.г.с.": segments_str,
    "m_i": m,
    "np_i": np.round(P * N, 2)
}).T
df.columns = [str(i) for i in range(10)]
display(df)

if p_value_chi2_b < alpha_level:
    print(f"p-value ({p_value_chi2_b:.5f}) < α ({alpha_level}) ==> H0 ОТВЕРГАЕТСЯ")
else:
    print(f"p-value ({p_value_chi2_b:.5f}) >= α ({alpha_level}) ==> Нет оснований отвергать H0")

--- Нормальное распределение (Пирсон) ---
Оценки ОМПГ: μ ≈ 5.2897, σ ≈ 2.6795
Статистика Δ ≈ 9.8026, p-value ≈ 0.20004



,0,1,2,3,4,5,6,7,8,9
Интервал п.г.с.,(-∞; 1),[1; 2),[2; 3),[3; 4),[4; 5),[5; 6),[6; 7),[7; 8),[8; 9),[9; +∞)
m_i,5,8,6,12,14,18,11,6,13,7
np_i,5.47,5.51,8.66,11.87,14.18,14.76,13.38,10.58,7.28,8.31


p-value (0.20004) >= α (0.05) ==> Нет оснований отвергать H0


In [7]:
bootstrap_iterations = 50_000

x_vals = np.arange(10)

delta_wave = np.sqrt(N) * np.max([
    max(np.abs(stats.norm.cdf(i, alpha_param, sigma_param) - F_emp[i]),
        np.abs(stats.norm.cdf(i, alpha_param, sigma_param) - F_emp[i+1]))
    for i in x_vals
])

bootstrap_delta = np.zeros(bootstrap_iterations)

# Бутстрап
for i in range(bootstrap_iterations):
    random_sample = np.random.normal(alpha_param, sigma_param, N)
    
    m_boot, _ = np.histogram(random_sample, bins=bounds)
    
    # Оцениваем параметры для бутстрап-выборки
    def neg_log_lik_boot(params):
        mu, sigma = params
        if sigma <= 0: return np.inf
        p = stats.norm.cdf(bounds[1:], mu, sigma) - stats.norm.cdf(bounds[:-1], mu, sigma)
        p = np.clip(p, 1e-12, 1.0)
        return -np.sum(m_boot * np.log(p))
    
    boot_guess = [np.mean(random_sample), np.std(random_sample, ddof=1)]
    res_boot = minimize(neg_log_lik_boot, boot_guess, method='Nelder-Mead')
    alpha_boot, sigma_boot = res_boot.x
    
    F_bootstrap_emp = np.array([sum(m_boot[:j]) for j in range(len(m_boot) + 1)]) / N
    
    sup = np.max([
        max(np.abs(stats.norm.cdf(j, alpha_boot, sigma_boot) - F_bootstrap_emp[j]),
            np.abs(stats.norm.cdf(j, alpha_boot, sigma_boot) - F_bootstrap_emp[j+1]))
        for j in x_vals
    ])
    
    bootstrap_delta[i] = np.sqrt(N) * sup

p_value_ks_b = np.sum(bootstrap_delta >= delta_wave) / bootstrap_iterations

print("--- Нормальное распределение (Колмогоров, Параметрический бутстрап) ---")
print(f"Статистика Δ (оригинал) = {delta_wave:.4f}")
print(f"p-value: {p_value_ks_b:.5f}\n")

if p_value_ks_b < alpha_level:
    print(f"p-value ({p_value_ks_b:.5f}) < α ({alpha_level}) ==> H0 ОТВЕРГАЕТСЯ")
else:
    print(f"p-value ({p_value_ks_b:.5f}) >= α ({alpha_level}) ==> Нет оснований отвергать H0")

--- Нормальное распределение (Колмогоров, Параметрический бутстрап) ---
Статистика Δ (оригинал) = 1.7305
p-value: 0.35146

p-value (0.35146) >= α (0.05) ==> Нет оснований отвергать H0
